<a href="https://colab.research.google.com/github/habibarezq/ML-Assignments-25/blob/main/Assingment-4/notebooks/k-means_and_gmm_Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports


In [5]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from enum import Enum
from sklearn.preprocessing import StandardScaler
from google.colab import files
uploaded = files.upload()
from gmm import GMM,CovarianceType
from Autoencoder import Autoencoder
from kmeans import NumpyKMeans

Saving Autoencoder.py to Autoencoder (1).py
Saving gmm.py to gmm (1).py
Saving kmeans.py to kmeans (1).py


## data setting


In [6]:

np.random.seed(42)

# Example dataset (replace with your actual dataset)
X = np.random.randn(1000, 784)

bottleneck_sizes = [2, 5, 10, 15, 20]

# Store results
ae_results = {}
pca_results = {}

## Experiment 5: K-Means after Autoencoder

In [ ]:
for bsize in bottleneck_sizes:
    print(f"\nTraining Autoencoder with bottleneck size {bsize}...")
    ae = Autoencoder(input_dim=784, hidden_dims=[256,128,64], bottleneck_dim=bsize, activation='relu')
    ae.train(X, epochs=50, batch_size=64)

    # Encode using autoencoder
    Z_ae = ae.encode(X)

    # K-Means on AE features
    kmeans_ae = NumpyKMeans(n_clusters=10, init='k-means++', max_iter=300, tol=1e-4, random_state=42)
    kmeans_ae.fit(Z_ae)
    kmeans_ae_labels = kmeans_ae.labels_

    # Reconstruction loss
    X_hat = ae.decode(Z_ae)
    recon_loss = np.mean((X - X_hat)**2)

    # K-Means clustering score (silhouette)
    kmeans_score = silhouette_score(Z_ae, kmeans_ae_labels)

    ae_results[bsize] = {
        "recon_loss": recon_loss,
        "kmeans_silhouette": kmeans_score
    }

    # PCA for comparison
    pca = PCA(n_components=bsize)
    Z_pca = pca.fit_transform(X)

    kmeans_pca = NumpyKMeans(n_clusters=10, init='k-means++', max_iter=300, tol=1e-4, random_state=42)
    kmeans_pca.fit(Z_pca)

    X_hat_pca = pca.inverse_transform(Z_pca)
    recon_loss_pca = np.mean((X - X_hat_pca)**2)
    kmeans_score_pca = silhouette_score(Z_pca, kmeans_pca.labels_)

    pca_results[bsize] = {
        "recon_loss": recon_loss_pca,
        "kmeans_silhouette": kmeans_score_pca
    }


Training Autoencoder with bottleneck size 2...
Epoch 1/50, Loss: 1.1191
Epoch 2/50, Loss: 1.1194
Epoch 3/50, Loss: 1.1190
Epoch 4/50, Loss: 1.1192
Epoch 5/50, Loss: 1.1193
Epoch 6/50, Loss: 1.1188
Epoch 7/50, Loss: 1.1190
Epoch 8/50, Loss: 1.1191
Epoch 9/50, Loss: 1.1192
Epoch 10/50, Loss: 1.1188
Epoch 11/50, Loss: 1.1190
Epoch 12/50, Loss: 1.1188
Epoch 13/50, Loss: 1.1189
Epoch 14/50, Loss: 1.1192
Epoch 15/50, Loss: 1.1188
Epoch 16/50, Loss: 1.1190
Epoch 17/50, Loss: 1.1188
Epoch 18/50, Loss: 1.1191
Epoch 19/50, Loss: 1.1186
Epoch 20/50, Loss: 1.1187
Epoch 21/50, Loss: 1.1186
Epoch 22/50, Loss: 1.1189
Epoch 23/50, Loss: 1.1187
Epoch 24/50, Loss: 1.1186
Epoch 25/50, Loss: 1.1183
Epoch 26/50, Loss: 1.1189
Epoch 27/50, Loss: 1.1186
Epoch 28/50, Loss: 1.1189
Epoch 29/50, Loss: 1.1188
Epoch 30/50, Loss: 1.1190
Epoch 31/50, Loss: 1.1188
Epoch 32/50, Loss: 1.1184
Epoch 33/50, Loss: 1.1186
Epoch 34/50, Loss: 1.1188
Epoch 35/50, Loss: 1.1187
Epoch 36/50, Loss: 1.1188
Epoch 37/50, Loss: 1.1192

## Experiment 6: GMM after Autoencoder

In [ ]:

gmm_results = {}
gmm_pca_results = {}

for bsize in bottleneck_sizes:
    #  AE FEATURES
    Z_ae = ae.encode(X)

    #  ADD HERE (strongly recommended)
    Z_ae = StandardScaler().fit_transform(Z_ae)

    gmm_ae = GMM(
        n_components=10,
        cov_type=CovarianceType.FULL,
        tol=1e-4,
        max_iter=100
    )
    gmm_ae.fit(Z_ae)
    gmm_ae_labels = gmm_ae.predict(Z_ae)

    #  REPLACE HERE
    n_clusters_ae = len(np.unique(gmm_ae_labels))
    if n_clusters_ae > 1:
        gmm_score = silhouette_score(Z_ae, gmm_ae_labels)
    else:
        gmm_score = -1   # or np.nan

    gmm_results[bsize] = gmm_score

    #  PCA FEATURES
    pca = PCA(n_components=bsize)
    Z_pca = pca.fit_transform(X)

    # ➕ ADD HERE (recommended)
    Z_pca = StandardScaler().fit_transform(Z_pca)

    gmm_pca = GMM(
        n_components=10,
        cov_type=CovarianceType.FULL,
        tol=1e-4,
        max_iter=100
    )
    gmm_pca.fit(Z_pca)
    gmm_pca_labels = gmm_pca.predict(Z_pca)

    #  REPLACE HERE
    n_clusters_pca = len(np.unique(gmm_pca_labels))
    if n_clusters_pca > 1:
        gmm_score_pca = silhouette_score(Z_pca, gmm_pca_labels)
    else:
        gmm_score_pca = -1   # or np.nan

    gmm_pca_results[bsize] = gmm_score_pca


## Autoencoder vs PCA: K-Means Results

In [ ]:
for bsize in bottleneck_sizes:
    print(f"Bottleneck {bsize}: AE Loss={ae_results[bsize]['recon_loss']:.4f}, AE Silhouette={ae_results[bsize]['kmeans_silhouette']:.4f}, "
          f"PCA Loss={pca_results[bsize]['recon_loss']:.4f}, PCA Silhouette={pca_results[bsize]['kmeans_silhouette']:.4f}")

## Autoencoder vs PCA: GMM Results

In [ ]:

for bsize in bottleneck_sizes:
    print(f"Bottleneck {bsize}: AE Silhouette={gmm_results[bsize]:.4f}, PCA Silhouette={gmm_pca_results[bsize]:.4f}")
